# 01 多个输入：两个标量 vs 打包成 shape (2,) 的 x

观察同一个预测函数，分别用两种方式接收两个输入：
- 方式A：预测函数接收**两个标量参数** `predict(x1, x2)`
- 方式B：把同一个样本的两个值打包成一个 **shape 为 (2,) 的张量** `x`，再传入 `predict(x)`

比较两者的预测值、损失与 shape。本题不写训练循环。

In [1]:
import torch

# 固定一组权重和偏置，只做前向观察，不需要梯度
w1, w2, b = 0.5, -1.0, 0.2

# 方式A：预测函数接收两个标量输入
def predict_scalar(x1, x2):
    return w1 * x1 + w2 * x2 + b

# 方式B：预测函数接收一个 shape 为 (2,) 的张量 x
def predict_vec(x):
    return w1 * x[0] + w2 * x[1] + b  # x[0]=第一个特征, x[1]=第二个特征

# 一个样本的两个特征，以及真实标签
x1, x2 = 3.0, 1.0
y_true = 1.0

In [2]:
# 方式A：直接传两个标量
y_pred_a = predict_scalar(x1, x2)

# 方式B：先把两个值打包成 shape (2,) 的张量
x = torch.tensor([x1, x2])
print('x =', x)
print('x.shape =', x.shape)
y_pred_b = predict_vec(x)

print('方式A 预测值 =', y_pred_a)
print('方式B 预测值 =', y_pred_b)
print('两者是否相等 =', abs(float(y_pred_a) - float(y_pred_b)) < 1e-6)

x = tensor([3., 1.])
x.shape = torch.Size([2])
方式A 预测值 = 0.7
方式B 预测值 = tensor(0.7000)
两者是否相等 = True


In [3]:
# 损失对比（均方误差）
loss_a = (y_pred_a - y_true) ** 2
loss_b = (y_pred_b - y_true) ** 2
print('方式A 损失 =', loss_a)
print('方式B 损失 =', float(loss_b))

方式A 损失 = 0.09000000000000002
方式B 损失 = 0.09000000357627869


## 观察

1. **预测值**：方式A 和方式B 算出的预测值完全相同（都是 `0.5*3.0 + (-1.0)*1.0 + 0.2 = 0.7`）。
2. **损失**：两种方式的均方误差也相同（`(0.7 - 1.0)^2 = 0.09`）。
3. **shape**：打包后的 `x` 形状是 `torch.Size([2])`，即一维张量、长度为 2；`x[0]` 对应第一个特征，`x[1]` 对应第二个特征。
4. **结论**：把两个标量打包成一个长度为 2 的张量再传进去，和分开传两个标量，**在数学上完全等价**，只是数据组织形式不同。后续多样本训练时，用向量/矩阵形式组织输入更方便批量计算。